In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# New Section

In [3]:
# This cell loads the prepared AMI baseline dataset from the existing project structure.

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
BASELINE_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "ami_baseline_dataset.csv"
)

print("Project directory:", PROJECT_DIR)
print("Baseline dataset:", BASELINE_PATH)
print("Dataset exists:", os.path.exists(BASELINE_PATH))

if not os.path.exists(BASELINE_PATH):
    raise FileNotFoundError(f"Dataset not found: {BASELINE_PATH}")

baseline_df = pd.read_csv(
    BASELINE_PATH,
    encoding="utf-8"
)

print("\nDataset loaded successfully.")
print("Shape:", baseline_df.shape)
print("Columns:", baseline_df.columns.tolist())
print("Meetings:", baseline_df["meeting_id"].tolist())

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Baseline dataset: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv
Dataset exists: True

Dataset loaded successfully.
Shape: (10, 6)
Columns: ['meeting_id', 'transcript', 'reference_summary', 'clean_transcript', 'timestamped_words', 'clean_words']
Meetings: ['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']


In [4]:
# This cell compares the raw and existing cleaned transcript for ES2004a before we send it to the summarization model.

meeting_id = "ES2004a"

row = baseline_df.loc[
    baseline_df["meeting_id"] == meeting_id
].iloc[0]

print("=" * 70)
print("RAW TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["clean_transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN WORDS — FIRST 1000 CHARACTERS")
print("=" * 70)
print(str(row["clean_words"])[:1000])

RAW TRANSCRIPT — FIRST 1500 CHARACTERS
[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit
[14.15 - 14.53] B: better
[14.53 - 14.53] B: ?
[17.88 - 18.15] A: Yeah
[18.15 - 18.15] A: .
[18.87 - 19.70] B: Okay
[19.70 - 19.70] B: ,
[19.70 - 19.99] B: that's
[19.99 - 20.29] B: fine
[20.29 - 20.29] B: .
[22.37 - 22.50] B: Am
[22.50 - 22.56] B: I
[22.56 - 22.78] B: supposed
[22.78 - 22.84] B: to
[22.84 - 22.90] B: be
[22.90 - 23.28] B: standing
[23.28 - 23.44] B: up
[23.44 - 23.81] B: there
[23.81 - 23.81] B: ?
[25.15 - 25.23] D: So
[25.18 - 25.60] B: Okay
[25.23 - 25.33] D: we've
[25.33 - 25.4

In [5]:
# This cell reconstructs the word-level AMI transcript into readable speaker utterances for BART input.

import re
import pandas as pd

def reconstruct_transcript(clean_transcript):
    """
    Convert AMI word-level speaker annotations into readable text.

    Example:
        B: Hello
        B: everybody
        A: Hi
        A: ,
        A: good
        A: morning
        B: .

    becomes:
        B: Hello everybody.
        A: Hi, good morning.
    """

    if not isinstance(clean_transcript, str):
        return ""

    lines = clean_transcript.splitlines()

    utterances = []
    current_speaker = None
    current_words = []

    punctuation = {".", ",", "?", "!", ";", ":"}

    def flush_utterance():
        nonlocal current_speaker, current_words

        if not current_words:
            return

        text = ""

        for word in current_words:
            word = word.strip()

            if not word:
                continue

            if word in punctuation:
                text = text.rstrip() + word
            else:
                if text:
                    text += " "
                text += word

        text = re.sub(r"\s+", " ", text).strip()

        if text:
            utterances.append(f"{current_speaker}: {text}")

        current_words = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        match = re.match(r"^([A-Z]):\s*(.*)$", line)

        if not match:
            continue

        speaker = match.group(1)
        word = match.group(2).strip()

        # Speaker changed → finish previous utterance
        if current_speaker is not None and speaker != current_speaker:
            flush_utterance()

        current_speaker = speaker

        if word:
            current_words.append(word)

        # Sentence-ending punctuation → finish utterance
        if word in {".", "?", "!"}:
            flush_utterance()

    # Add final unfinished utterance
    flush_utterance()

    return "\n".join(utterances)


# Test on ES2004a
test_clean = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "clean_transcript"
].iloc[0]

bart_test = reconstruct_transcript(test_clean)

print("=" * 70)
print("BART-READY TRANSCRIPT — ES2004a")
print("=" * 70)
print(bart_test[:3000])

BART-READY TRANSCRIPT — ES2004a
A: Hmm hmm hmm.
B: Are we we're not allowed to dim the lights so people can see that a bit better?
A: Yeah.
B: Okay, that's fine.
B: Am I supposed to be standing up there?
D: So
B: Okay
D: we've got both
B: .
D: of these clipped on?
D: She gonna answer me
B: Yeah
D: or not
B: , I've got
D: ?
D: Right, both of them, okay.
B: Yes.
D: God.
D: Jesus, it's gonna fall off.
A: Okay.
A: Yep, yep.
A: Okay.
B: Okay
A: Tu tu tu tu
B: .
B: Hello everybody
A: Hi, good morning
B: .
B: Um
A: .
B: I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough.
B: Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other.
B: Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand.
B: Now, we're developing a remote control which you probably already know.
B: Um, we

In [6]:
# This cell creates a speaker-neutral BART input while preserving the chronological content of the AMI transcript.

def create_bart_input(reconstructed_transcript):
    """
    Remove AMI speaker labels while preserving the reconstructed
    chronological transcript.
    """

    if not isinstance(reconstructed_transcript, str):
        return ""

    lines = reconstructed_transcript.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Remove speaker label such as A:, B:, C:, D:
        line = re.sub(r"^[A-Z]:\s*", "", line)

        # Normalize whitespace
        line = re.sub(r"\s+", " ", line).strip()

        if line:
            cleaned_lines.append(line)

    # Join utterances into paragraphs
    text = " ".join(cleaned_lines)

    # Normalize spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


bart_input_test = create_bart_input(bart_test)

print("=" * 70)
print("SPEAKER-NEUTRAL BART INPUT — ES2004a")
print("=" * 70)
print(bart_input_test[:3000])

print("\n" + "=" * 70)
print("WORD COUNT")
print("=" * 70)
print(len(bart_input_test.split()))

SPEAKER-NEUTRAL BART INPUT — ES2004a
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna fall off. Okay. Yep, yep. Okay. Okay Tu tu tu tu. Hello everybody Hi, good morning. Um. I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough. Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other. Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand. Now, we're developing a remote control which you probably already know. Um, we want it to be original, something that's uh people haven't thought of, that's not out in the shops, um, t

In [7]:
# This cell creates BART-ready inputs for all meetings and checks their lengths before chunking.

baseline_df["bart_input"] = baseline_df["clean_transcript"].apply(
    lambda x: create_bart_input(reconstruct_transcript(x))
)

baseline_df["bart_input_words"] = baseline_df["bart_input"].apply(
    lambda x: len(x.split())
)

print("=" * 70)
print("BART INPUT LENGTHS")
print("=" * 70)

print(
    baseline_df[
        ["meeting_id", "bart_input_words"]
    ].to_string(index=False)
)

print("\nAverage BART input words:",
      round(baseline_df["bart_input_words"].mean()))

print("Maximum BART input words:",
      baseline_df["bart_input_words"].max())

print("Minimum BART input words:",
      baseline_df["bart_input_words"].min())

BART INPUT LENGTHS
meeting_id  bart_input_words
   ES2004a              2614
   ES2004b              6763
   ES2004c              6968
   ES2004d              6134
   ES2005a               747
   ES2005b              6190
   ES2005c              6694
   ES2006a              2777
   ES2006b              6201
   ES2008a              2506

Average BART input words: 4759
Maximum BART input words: 6968
Minimum BART input words: 747


In [8]:
# This cell splits each BART-ready meeting transcript into overlapping chunks that can be processed safely by BART.

CHUNK_WORDS = 900
OVERLAP_WORDS = 100


def chunk_text(text, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    """Split text into overlapping word-based chunks."""

    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap_words

    return chunks


# Test chunking on ES2004a
test_chunks = chunk_text(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0]
)

print("=" * 70)
print("CHUNKING TEST — ES2004a")
print("=" * 70)

print("Total words:", len(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0].split()
))

print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):
    print(f"\nChunk {i}: {len(chunk.split())} words")
    print(chunk[:300] + "...")

CHUNKING TEST — ES2004a
Total words: 2614
Number of chunks: 4

Chunk 1: 900 words
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 words
Uh. It's not a vampire bat honestly Okay, yeah.. Uh and somewhere there's a body behind Okay. That's, some my dreadful sort of that's the worst yet bird, that's. it's meant to be an eagle A seagu Ah Eagle right eagle, okay, right., not you okay can a seagull tell. it's a flying animal could. have be...

Chunk 3: 900 words
they're. when you've got the main things on the front of it and a section opens up or something to the other functions where you can do sound or options Oh yeah kind. of recording, things like that inside it Mm-hmm.. 'Cause it doesn't make when you pick it up it doesn't

In [9]:
# This cell loads the BART summarization model on the available Tesla T4 GPU for a single-chunk baseline test.

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(DEVICE)

bart_model.eval()

print("BART loaded successfully.")

Model: facebook/bart-large-cnn
Device: cuda


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART loaded successfully.


In [10]:
# This cell generates a first BART summary from one ES2004a transcript chunk to verify the model output before processing all meetings.

import torch

test_chunk = test_chunks[0]

inputs = tokenizer(
    test_chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():
    summary_ids = bart_model.generate(
        **inputs,
        max_length=180,
        min_length=60,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

test_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("BART TEST SUMMARY — ES2004a CHUNK 1")
print("=" * 70)
print(test_summary)

BART TEST SUMMARY — ES2004a CHUNK 1
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on a white board. The group then discuss their ideas and work on the design. The project is due to be completed by the end of the year.


In [12]:
# This cell creates stable BART token chunks without altering BPE spacing during decoding.

BART_CHUNK_TOKENS = 900
BART_OVERLAP_TOKENS = 100


def chunk_text_by_tokens(
    text,
    tokenizer,
    chunk_tokens=BART_CHUNK_TOKENS,
    overlap_tokens=BART_OVERLAP_TOKENS
):
    """Create overlapping chunks directly from BART token IDs."""

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if not token_ids:
        return []

    chunks = []
    start = 0

    while start < len(token_ids):

        end = min(
            start + chunk_tokens,
            len(token_ids)
        )

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        chunks.append({
            "text": chunk_text,
            "token_count": len(chunk_ids),
            "start_token": start,
            "end_token": end
        })

        if end >= len(token_ids):
            break

        start = end - overlap_tokens

    return chunks


# Test the corrected token-aware chunking on ES2004a.

es2004a_bart_input = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "bart_input"
].iloc[0]

token_chunks = chunk_text_by_tokens(
    es2004a_bart_input,
    tokenizer
)

print("=" * 70)
print("CORRECTED TOKEN-AWARE CHUNKING — ES2004a")
print("=" * 70)

total_tokens = len(
    tokenizer.encode(
        es2004a_bart_input,
        add_special_tokens=False
    )
)

print("Total input tokens:", total_tokens)
print("Number of chunks:", len(token_chunks))

for i, chunk in enumerate(token_chunks, 1):
    print(
        f"\nChunk {i}: "
        f"{chunk['token_count']} tokens "
        f"({chunk['start_token']} → {chunk['end_token']})"
    )
    print(chunk["text"][:300] + "...")

CORRECTED TOKEN-AWARE CHUNKING — ES2004a
Total input tokens: 3463
Number of chunks: 5

Chunk 1: 900 tokens (0 → 900)
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 tokens (800 → 1700)
gonna be because that looks like a beak now, so. Crocodile? Gonna be Yeah a, it bird can be. a crocodile, it can be Is a it crocodile gonna be. Well it was it was it's an gonna at be first a bird firstly. it was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a croc...

Chunk 3: 900 tokens (1600 → 2500)
mm.. Um, and profit aim is fifty million Euros, which is uh In our first year? Yi yes, um yeah, I presume so Mm-hmm. So. Um then You've got market range international and you did say earlier it's got to be 

In [13]:
# This cell generates an abstractive BART summary for each token-aware ES2004a chunk.

def summarize_chunk(chunk_text):
    inputs = tokenizer(
        chunk_text,
        return_tensors="pt",
        max_length=1024,
        truncation=False
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        summary_ids = bart_model.generate(
            **inputs,
            max_length=180,
            min_length=40,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


chunk_summaries = []

print("=" * 70)
print("BART CHUNK SUMMARIZATION — ES2004a")
print("=" * 70)

for i, chunk in enumerate(token_chunks, 1):

    print(f"\nProcessing chunk {i}/{len(token_chunks)}...")

    summary = summarize_chunk(chunk["text"])

    chunk_summaries.append(summary)

    print(f"\nCHUNK {i} SUMMARY:")
    print(summary)

print("\n" + "=" * 70)
print("ALL CHUNK SUMMARIES GENERATED")
print("=" * 70)
print("Number of summaries:", len(chunk_summaries))

BART CHUNK SUMMARIZATION — ES2004a

Processing chunk 1/5...

CHUNK 1 SUMMARY:
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design.

Processing chunk 2/5...

CHUNK 2 SUMMARY:
It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros.

Processing chunk 3/5...

CHUNK 3 SUMMARY:
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation.

Processing chunk 4/5...

CHUNK 4 SUMMARY:
I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get

In [14]:
# This cell combines the five independently generated BART chunk summaries into one baseline meeting-level summary input.

combined_chunk_summary = " ".join(chunk_summaries)

print("=" * 70)
print("COMBINED CHUNK SUMMARIES — ES2004a")
print("=" * 70)

print(combined_chunk_summary)

print("\n" + "=" * 70)
print("COMBINED SUMMARY TOKEN COUNT")
print("=" * 70)

combined_tokens = tokenizer.encode(
    combined_chunk_summary,
    add_special_tokens=False
)

print("Tokens:", len(combined_tokens))

COMBINED CHUNK SUMMARIES — ES2004a
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design. It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros. The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get Yeah, you'd you'd have to keep it down to a black and white L_C_D_ thing anyway. The other thing is, just ch chucking into mobile phone f design features again, it could h

In [15]:
# This cell performs the second-stage BART summarization to create the final meeting-level baseline MoM for ES2004a.

final_inputs = tokenizer(
    combined_chunk_summary,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

final_inputs = {
    key: value.to(DEVICE)
    for key, value in final_inputs.items()
}

with torch.no_grad():
    final_summary_ids = bart_model.generate(
        **final_inputs,
        max_length=220,
        min_length=80,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

es2004a_baseline_summary = tokenizer.decode(
    final_summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("FINAL BART BASELINE — ES2004a")
print("=" * 70)
print(es2004a_baseline_summary)

FINAL BART BASELINE — ES2004a
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. It could have a flip top remote control so that when you flip over the top, your screen is you can have a bigger screen in Mm-hmm the. flip over. . Just. just a quick thing about Sure the. um about what you're saying about the uh does does it need to be fashionable?


Then we'll compare it with the AMI reference

The reference for ES2004a is:

The Project Manager gave an introduction to the goal of the project,
to create a trendy yet user-friendly remote.

She presented a long-range agenda for the whole project.

The group introduced themselves to each other and practiced with
the meeting room tools by drawing on the board.

The Project Manager presented the project budget, the projected
price point, and the projected profit aim for the project.

Then the group began a discussion about their own experiences with
remote controls to generate initial design ideas for making the
product user-friendly.

They discussed grouping features into a menu and adding an LCD display.

They also discussed the look of various materials that may be used
in the design, in keeping with the company's goal to create
fashionable electronics.

The important baseline question is now:

Can plain BART recover the important meeting-level information from the transcript?

In [16]:
# This cell records the exact baseline BART configuration so later improvements can be compared fairly.

BASELINE_CONFIG = {
    "model": "facebook/bart-large-cnn",
    "device": str(DEVICE),
    "chunk_tokens": BART_CHUNK_TOKENS,
    "overlap_tokens": BART_OVERLAP_TOKENS,
    "chunk_max_length": 180,
    "chunk_min_length": 40,
    "final_max_length": 220,
    "final_min_length": 80,
    "num_beams": 4,
    "length_penalty": 2.0,
    "no_repeat_ngram_size": 3,
    "dataset": "AMI",
    "meetings": len(baseline_df)
}

print("=" * 70)
print("BART BASELINE CONFIGURATION")
print("=" * 70)

for key, value in BASELINE_CONFIG.items():
    print(f"{key:25}: {value}")

BART BASELINE CONFIGURATION
model                    : facebook/bart-large-cnn
device                   : cuda
chunk_tokens             : 900
overlap_tokens           : 100
chunk_max_length         : 180
chunk_min_length         : 40
final_max_length         : 220
final_min_length         : 80
num_beams                : 4
length_penalty           : 2.0
no_repeat_ngram_size     : 3
dataset                  : AMI
meetings                 : 10


In [17]:
# This cell runs the same BART baseline pipeline on all 10 AMI meetings and saves each generated MoM.

import os
import json
import time

MOM_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "mom"
)

os.makedirs(MOM_OUTPUT_DIR, exist_ok=True)

all_baseline_results = []

print("=" * 70)
print("RUNNING BART BASELINE — ALL AMI MEETINGS")
print("=" * 70)

for idx, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]
    bart_input = row["bart_input"]

    print(
        f"\n[{idx + 1}/{len(baseline_df)}] "
        f"Processing {meeting_id}..."
    )

    start_time = time.time()

    # Create token-aware chunks
    meeting_chunks = chunk_text_by_tokens(
        bart_input,
        tokenizer
    )

    meeting_chunk_summaries = []

    # Generate summary for each chunk
    for chunk_idx, chunk in enumerate(meeting_chunks, 1):

        summary = summarize_chunk(
            chunk["text"]
        )

        meeting_chunk_summaries.append(summary)

    # Combine chunk summaries
    combined_summary = " ".join(
        meeting_chunk_summaries
    )

    # Token count of combined summaries
    combined_token_count = len(
        tokenizer.encode(
            combined_summary,
            add_special_tokens=False
        )
    )

    # Second-stage summarization
    final_inputs = tokenizer(
        combined_summary,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    final_inputs = {
        key: value.to(DEVICE)
        for key, value in final_inputs.items()
    }

    with torch.no_grad():

        final_summary_ids = bart_model.generate(
            **final_inputs,
            max_length=220,
            min_length=80,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    final_summary = tokenizer.decode(
        final_summary_ids[0],
        skip_special_tokens=True
    )

    elapsed = time.time() - start_time

    # Store complete result
    result = {
        "meeting_id": meeting_id,
        "model": "facebook/bart-large-cnn",
        "num_chunks": len(meeting_chunks),
        "combined_summary_tokens": combined_token_count,
        "chunk_summaries": meeting_chunk_summaries,
        "final_summary": final_summary,
        "processing_time_seconds": round(elapsed, 2)
    }

    all_baseline_results.append(result)

    # Save individual meeting result
    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"  Chunks: {len(meeting_chunks)}"
    )

    print(
        f"  Combined tokens: {combined_token_count}"
    )

    print(
        f"  Time: {elapsed:.1f} sec"
    )

    print(
        f"  Saved: {output_path}"
    )

print("\n" + "=" * 70)
print("BART BASELINE RUN COMPLETE")
print("=" * 70)

print(
    "Meetings processed:",
    len(all_baseline_results)
)

RUNNING BART BASELINE — ALL AMI MEETINGS

[1/10] Processing ES2004a...
  Chunks: 5
  Combined tokens: 327
  Time: 14.1 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004a_bart_baseline.json

[2/10] Processing ES2004b...
  Chunks: 11
  Combined tokens: 794
  Time: 19.2 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004b_bart_baseline.json

[3/10] Processing ES2004c...
  Chunks: 11
  Combined tokens: 823
  Time: 18.9 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004c_bart_baseline.json

[4/10] Processing ES2004d...
  Chunks: 10
  Combined tokens: 604
  Time: 15.0 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004d_bart_baseline.json

[5/10] Processing ES2005a...
  Chunks: 2
  Combined tokens: 158
  Time: 4.4 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2005a_bart_baseline.json

[6/10] Processing ES2005b...
  Chunks: 10
  Combined tokens: 661
 

In [18]:
# This cell checks whether the Hugging Face Evaluate library is available for computing ROUGE scores.

import importlib.util

evaluate_available = (
    importlib.util.find_spec("evaluate") is not None
)

print("=" * 70)
print("EVALUATION LIBRARY CHECK")
print("=" * 70)

print("Evaluate installed:", evaluate_available)

EVALUATION LIBRARY CHECK
Evaluate installed: False


In [19]:
# This cell installs the Hugging Face evaluation package and ROUGE dependency required for baseline evaluation.

!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [20]:
# This cell verifies that the evaluation library and ROUGE metric can now be imported successfully.

import evaluate

print("=" * 70)
print("EVALUATION LIBRARY READY")
print("=" * 70)

print("Evaluate version:", evaluate.__version__)

rouge = evaluate.load("rouge")

print("ROUGE metric loaded successfully.")

EVALUATION LIBRARY READY
Evaluate version: 0.4.6


ROUGE metric loaded successfully.


In [21]:
# This cell computes ROUGE-1, ROUGE-2, and ROUGE-L for all 10 BART-generated MoMs against the AMI reference summaries.

import os
import json
import pandas as pd

rouge_results = []

print("=" * 70)
print("BART BASELINE — ROUGE EVALUATION")
print("=" * 70)

for _, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]

    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:
        result = json.load(f)

    generated_summary = result["final_summary"]
    reference_summary = row["reference_summary"]

    scores = rouge.compute(
        predictions=[generated_summary],
        references=[reference_summary],
        use_stemmer=True
    )

    rouge_results.append({
        "meeting_id": meeting_id,
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

    print(
        f"{meeting_id}: "
        f"R1={scores['rouge1']:.4f} | "
        f"R2={scores['rouge2']:.4f} | "
        f"RL={scores['rougeL']:.4f}"
    )


rouge_df = pd.DataFrame(rouge_results)

print("\n" + "=" * 70)
print("AVERAGE BASELINE ROUGE")
print("=" * 70)

print(
    f"ROUGE-1: {rouge_df['rouge1'].mean():.4f}"
)

print(
    f"ROUGE-2: {rouge_df['rouge2'].mean():.4f}"
)

print(
    f"ROUGE-L: {rouge_df['rougeL'].mean():.4f}"
)

BART BASELINE — ROUGE EVALUATION
ES2004a: R1=0.2247 | R2=0.0226 | RL=0.1273
ES2004b: R1=0.1671 | R2=0.0148 | RL=0.1032
ES2004c: R1=0.2143 | R2=0.0240 | RL=0.1310
ES2004d: R1=0.1758 | R2=0.0239 | RL=0.0998
ES2005a: R1=0.1949 | R2=0.0104 | RL=0.1333
ES2005b: R1=0.2382 | R2=0.0891 | RL=0.1496
ES2005c: R1=0.2135 | R2=0.0524 | RL=0.1406
ES2006a: R1=0.2126 | R2=0.0462 | RL=0.1264
ES2006b: R1=0.1818 | R2=0.0400 | RL=0.0966
ES2008a: R1=0.1720 | R2=0.0432 | RL=0.1075

AVERAGE BASELINE ROUGE
ROUGE-1: 0.1995
ROUGE-2: 0.0367
ROUGE-L: 0.1215


In [22]:
# This cell saves the BART baseline ROUGE scores for use in later comparisons and the final project evaluation.

EVALUATION_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

os.makedirs(
    EVALUATION_DIR,
    exist_ok=True
)

ROUGE_OUTPUT_PATH = os.path.join(
    EVALUATION_DIR,
    "bart_baseline_rouge.csv"
)

rouge_df.to_csv(
    ROUGE_OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("BASELINE EVALUATION SAVED")
print("=" * 70)
print(ROUGE_OUTPUT_PATH)

BASELINE EVALUATION SAVED
/content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results/bart_baseline_rouge.csv
